# 02 — Clean & Quality Check

> **AI-Assisted Development** — This project was built with [Kiro](https://kiro.dev). See `SOURCES.md` for full attribution.

All cleaning operations are SQL-driven inside DuckDB. This notebook documents
what was done and lets you inspect the results.

### Cleaning operations performed:
1. Renamed FARS `alcohol_fatalities` → `impaired_fatalities_any` (includes drugs/meds, not just alcohol)
2. Verified state name consistency across all 14 source tables
3. Parsed NCSL criminal status text → `first_offense_felony` flag + `felony_threshold`
4. Extracted FARS license status (suspended/revoked drivers in fatal crashes)
5. Ran quality checks on all tables

### Key decisions:
- **FARS 2021+ vs 2015–2020:** Not directly comparable due to schema change. Use `impairment_method` column to filter.
- **For published alcohol-specific analysis:** Use `nhtsa_imputed_2024` (BAC≥.08 only, statistically modeled).
- **FARS `drimpair=9`** captures alcohol + drugs + medication — broader than alcohol alone.
- **felony_threshold = NULL** means DUI is always a misdemeanor in that state (CA, DC, NJ, etc.)

In [ ]:
import sys
sys.path.insert(0, "..")

import duckdb
import pandas as pd
from src.ingest import load_config

cfg = load_config("../config.yaml")
con = duckdb.connect(str("../" + cfg['settings']['duckdb_file']))
print(f"Tables: {len(con.execute('SHOW TABLES').df())}")
con.execute('SHOW TABLES').df()

---
## 1. FARS trends — corrected column names
The raw FARS `drimpair=9` code captures **any impairment** (alcohol, drugs, medication).
Columns renamed from `alcohol_*` to `impaired_*_any` to avoid overstating alcohol's role.

An `impairment_method` flag distinguishes the 2015–2020 methodology (`drunk_dr` field in accident.csv)
from 2021+ (`drimpair` code 9 in a separate file). These are **not directly comparable**.

In [ ]:
# National totals by year — note the methodology break at 2021
national = con.execute("""
    SELECT year, impairment_method,
           SUM(total_fatalities) AS total_fatalities,
           SUM(impaired_fatalities_any) AS impaired_fatalities,
           ROUND(SUM(impaired_fatalities_any) * 100.0 / SUM(total_fatalities), 1) AS pct_impaired
    FROM fars_trends_clean
    GROUP BY year, impairment_method
    ORDER BY year
""").df()
national

---
## 2. State name consistency
All tables use canonical Census Bureau state names (50 states + DC).
NCSL territories (American Samoa, Guam, Puerto Rico, USVI) were filtered out.

In [ ]:
# Verify: all key tables have exactly 51 rows
for table in ['nhtsa_imputed_2024', 'alcohol_consumption', 'dui_arrests_2023',
              'dui_enforcement', 'dui_criminal_status_clean', 'fars_license_status_2024']:
    n = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'  {table}: {n} rows {"✓" if n == 51 else "⚠"}')

---
## 3. Criminal status — felony flags
Parsed from NCSL free-text `criminal_status` column.

- `first_offense_felony`: 1 if any first DUI offense *can* be charged as felony
- `felony_threshold`: which offense number triggers felony (NULL = always misdemeanor)

In [ ]:
felony = con.execute("""
    SELECT state_name, first_offense_felony, felony_threshold, criminal_status
    FROM dui_criminal_status_clean
    ORDER BY felony_threshold, state_name
""").df()
print(f"First offense felony: {felony['first_offense_felony'].sum():.0f} states")
print(f"Threshold distribution: {felony['felony_threshold'].value_counts().sort_index().to_dict()}")
print(f"Always misdemeanor (NULL threshold): {felony['felony_threshold'].isna().sum()} states")
felony

---
## 4. FARS license status — suspended drivers in fatal crashes
From `vehicle.csv` field `L_STATUS`: 1=Suspended, 2=Revoked.

Cross-referenced with `drimpair.csv` to find how many suspended-license drivers
were also impaired at the time of the fatal crash.

**Key finding:** 3,095 drivers in 2024 fatal crashes had suspended/revoked licenses.
22.6% of those were also impaired — suggesting suspension alone doesn't prevent impaired driving.

In [ ]:
lic = con.execute("""
    SELECT state_name, total_drivers_in_fatal_crashes,
           drivers_suspended_revoked, pct_suspended,
           drivers_suspended_and_impaired, pct_suspended_who_impaired
    FROM fars_license_status_2024
    ORDER BY pct_suspended DESC
""").df()
print(f"National: {lic['drivers_suspended_revoked'].sum():,} suspended/revoked in fatal crashes")
print(f"Of those also impaired: {lic['drivers_suspended_and_impaired'].sum():,} ({lic['drivers_suspended_and_impaired'].sum()/lic['drivers_suspended_revoked'].sum()*100:.1f}%)")
lic

---
## 5. Quality summary
All cleaned tables: 51 rows, no duplicates, minimal nulls.

In [ ]:
# Quick quality check on all cleaned tables
clean_tables = ['nhtsa_imputed_2024', 'alcohol_consumption', 'dui_arrests_2023',
                'dui_enforcement', 'dui_criminal_status_clean', 'fars_license_status_2024',
                'fars_trends_clean', 'fars_2024_clean']
for t in clean_tables:
    df = con.execute(f'SELECT * FROM {t}').df()
    nulls = df.isnull().sum().sum()
    print(f'  {t}: {len(df)} rows, {nulls} nulls')

---
**Next:** open `03-prepare.ipynb` to compute per-capita rates, build the master
analysis table, and package the sellable dataset.

In [ ]:
con.close()